# Scopus Content 2LS — was the ticket really escalated, how often did it bounce, and what did it cost the customer?

Every ticket in `data/scopus content/scopus content 2ls verbatim cleaned 2025_2026.xlsx` is supposed
to be a 1LS → 2LS escalation. This notebook reads each transcript and decides, with a verbatim quote
and a justification for every judgement:

1. **Was it actually escalated to 2LS?** And to whom (technical/second-line team, content supplier, other team).
2. **How many times did it go back and forth?** Hand-offs 1LS → 2LS, and returns 2LS → 1LS
   (answers, rejections, requests for more information), as an ordered event list.
3. **Were there agent / private notes about the escalation?**
4. **Customer effort**: effort score, friction points, chasing, repeated information, outcome,
   transparency, end sentiment, deadline pressure.

Scope: `IS_VALUE_ADD == TRUE` and `_merge == "both"` → **63,824 tickets**. The free rule-based
features (sections 2–3) cover all of them. The LLM reads a **stratified sample of ~5,760 tickets**, and
every population figure is weighted back to the 63,824 (see *Sampling design*). Running the LLM on
everything would cost ~$2,000 and take ~30 hours.

## What profiling the data found, and why the notebook is shaped this way

**The transcripts are rebuilt from the raw TSV, not taken from the xlsx.** The `Conversation` column
in the xlsx repeats messages 2–4 times: about 10% of rows in the raw thread export
(`scopus content 2ls verbatim 2025_2026.tsv`) are duplicates on `(INCIDENT_ID, THREAD_ID, sequence)`, a
join fan-out upstream. The xlsx also drops the per-message timestamp and agent ID, and 241
conversations are cut at Excel's 32,767-character cell limit. The raw TSV is deduplicated here and
every turn keeps its date and an anonymised agent tag, so the model can see waits and agent changes.

**There is no separate private-note entry type.** The raw export has only `Customer`,
`Customer Proxy` and `Staff Account` entries. Internal notes still show up, but written into staff
entries: "This should be tracked closely … a dedicated group looking into this on both 1LS and 2LS",
or agents telling the customer "I left a priority private note for our second-line support team".
They are found by reading the text: a regex flag here, and a judgement from the LLM.

**Automated messages are identified by their text, not their sender.** Two shared staff accounts send
the "ticket will be assigned to a Content Service Desk specialist" acknowledgement and the "confirming
that it remains open … our technical team is diligently working" reminder, but those same accounts also
send real replies. The templates are collapsed to a one-line `System (automated)` turn, so they neither
read as an escalation nor inflate the transcript.

**"I have initiated the process to add your paper to our database" counts as an escalation.** This
agent template names no team, but it means the case was sent for processing. It is the only hand-off
wording on ~14,300 tickets (22%), so it moves the headline figure by about that much. Decided with the
business, not inferred. Both the prompt and the `rx_*` regex treat it as a hand-off.

**Raw entries are capped at 9,000 characters at source.** Very long single messages (pasted article
lists, forwarded chains) are therefore already incomplete before this notebook sees them.

## What this notebook does not tell you

- It judges **what the transcript shows**. A hand-off to 2LS done silently in the ticketing system, with no
  trace in any message, reads as "not escalated" or "unclear". The `Incident Has Child or Task` flag is
  `Yes` on every row, so the metadata cannot separate the two either.
- `obs_repeat_contact` is computed **within this 2LS population only**, not across all Scopus contacts.
- Effort is the model's reading of the transcript. It is triangulated against the observed `obs_*`
  measures after the sample run, but there is no survey ground truth here.
- LLM figures are **sample estimates**. A year × contact reason × resolution code cell is only
  reported as `reportable` when it has at least 50 tickets in the population and 30 in the sample.
  Smaller cells are shown, but they are indicative only.

## Cost gates
| Gate | Tickets | Rough cost | Flag |
|---|---|---|---|
| Sections 1–4 (data, rule features, sampling frame, build) | all | free | — |
| Gate 1: smoke test | 40 (from the sample) | ~$1.30 | always runs |
| Sample run | ~5,760 | ~$180, ~2.7 h at $0.031/ticket | `RUN_SAMPLE` |

## 0. Configuration — edit this cell

In [ ]:
import hashlib
import html
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# -- Project root discovery (walk up until rca_pipeline/ is found) ----------
_root = Path().absolute()
while not (_root / "definitions").is_dir() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
BASE = _root

# The system Python has pandas built against NumPy 1.x and fails on import.
assert pd.__version__.startswith('3.'), (
    f'pandas {pd.__version__} - expected 3.x from .venv. Wrong kernel selected?')

# -- Inputs ----------------------------------------------------------------
DATA_DIR     = BASE / 'data' / 'scopus content'
TICKETS_XLSX = DATA_DIR / 'scopus content 2ls verbatim cleaned 2025_2026.xlsx'  # filter + metadata
RAW_TSV      = DATA_DIR / 'scopus content 2ls verbatim 2025_2026.tsv'           # the transcripts
EXPECTED_TICKETS = 63_824      # IS_VALUE_ADD == TRUE & _merge == 'both', measured at profiling
for _p in (TICKETS_XLSX, RAW_TSV):
    assert _p.exists(), f'missing input: {_p}'

# -- Build artefacts -------------------------------------------------------
DEFN_XLSX     = BASE / 'definitions' / 'scopus_2ls_2026_pipeline_definition.xlsx'
TAXONOMY_DIR  = BASE / 'generated' / 'taxonomies'
MODULES_DIR   = BASE / 'generated' / 'modules'
CONFIG_JSON   = BASE / 'generated' / 'pipeline_config.json'
CONFIG_BACKUP = BASE / 'generated' / '_backup_pre_scopus_2ls'

# -- Columns ---------------------------------------------------------------
ID_COL   = 'INCIDENT_REF'
TEXT_COL = 'Conversation'
CUSTOMER_TYPES = ('Customer', 'Customer Proxy')
STAFF_TYPE     = 'Staff Account'

# -- The module ------------------------------------------------------------
MOD    = 's2ls_escalation_effort'
PREFIX = 'SLJ'          # Second-Line Journey
STUB   = 's2'           # every LLM field is s2_*, clear of _STRUCTURAL_PREFIXES ('escalat', 'resolution_', 'urgency')

# -- Preprocessing ---------------------------------------------------------
TURN_CAP   = 4_000      # one message longer than this keeps head + tail
TURN_HEAD  = 1_500
TRUNC_CAP  = 24_000     # whole-transcript ceiling sent to the model
TRUNC_HEAD = 6_000
TRUNC_TAIL = 17_900     # the marker counts toward the cap

# -- LLM tuning ------------------------------------------------------------
# batch_size 1, not the usual 5. The first smoke test ran at 3 and the model mixed
# the tickets up: 2 of 40 got another ticket's quote as evidence, and 2 plain
# "I have raised your request to our technical team" tickets came back "not
# escalated". One transcript per call costs more but cannot cross-contaminate.
TUNING = {
    'batch_size': 1, 'max_chars': 50_000, 'concurrency': 8,
    'retries': 5, 'max_tokens': 8192, 'track_costs': True,
    'cost_report_dir': 'runs/cost_reports',
    'temperature': 0.0,
}
# Settings that change what the model returns, and so invalidate a checkpoint.
OUTPUT_AFFECTING = ('temperature', 'max_tokens', 'batch_size', 'max_chars')

# -- Sampling --------------------------------------------------------------
# Strata: year x contact reason (lvl 1) x resolution code. Running the LLM on all
# 63,824 tickets would cost ~$2,000 and take ~30 h; this sample is ~5,760 (~$180, ~2.7 h).
CELL_N       = 50       # tickets per cell (whole cell if smaller). 30 -> ~4,000 tickets, +-18 pts per cell
COLLAPSE_MIN = 50       # contact reasons / resolution codes with fewer tickets become "Other"
REPORT_MIN_SAMPLED = 30 # a cell is 'reportable' with >= COLLAPSE_MIN population and this many sampled
COST_PER_TICKET, TICKETS_PER_SEC = 0.031, 0.6    # measured at the smoke test, for the estimate only

# -- Gates -----------------------------------------------------------------
SMOKE_N, SEED = 40, 0
RUN_SAMPLE = True      # set True after reading the smoke output (gate 1) - launches the ~$180 run

# -- Outputs ---------------------------------------------------------------
INTER_DIR      = BASE / 'runs' / 'intermediate'
SMOKE_DIR      = BASE / 'runs' / 'smoke_s2ls'
OUT_DIR        = BASE / 'runs' / 'output'
CHECKPOINT_DIR = BASE / 'runs' / 'checkpoints'
PROFILE_DIR    = BASE / 'profiles'
for _d in (INTER_DIR, SMOKE_DIR, OUT_DIR, CHECKPOINT_DIR, PROFILE_DIR, DEFN_XLSX.parent):
    _d.mkdir(parents=True, exist_ok=True)

# s2ls_* stems so no checkpoint can collide with another notebook's
TRANSCRIPTS_CSV = INTER_DIR / 's2ls_transcripts.csv'        # id + rebuilt transcript, every ticket
FEATURES_CSV    = INTER_DIR / 's2ls_rule_features.csv'      # obs_* and rx_*, every ticket
SMOKE_INPUT     = SMOKE_DIR / 's2ls_smoke_input.csv'
SAMPLE_FRAME    = INTER_DIR / 's2ls_sample_frame.csv'       # sampled ids, cell, weight
SAMPLE_INPUT    = INTER_DIR / 's2ls_sample_input.csv'
PROFILE         = PROFILE_DIR / 'scopus_2ls_2026_escalation_effort.json'
SMOKE_OUT       = SMOKE_DIR / 's2ls_smoke_output.xlsx'
SMOKE_REVIEW    = SMOKE_DIR / 's2ls_smoke_review.xlsx'
SAMPLE_OUT      = OUT_DIR / 'scopus_2ls_2026_sample.xlsx'
FINAL_XLSX      = OUT_DIR / 'scopus_2ls_2026_escalation_effort_sample.xlsx'
FINAL_CSV       = OUT_DIR / 'scopus_2ls_2026_escalation_effort_sample_with_transcripts.csv'

# rca run prints non-cp1252 characters; force UTF-8 on every child process.
SUBPROCESS_ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8', 'PYTHONUTF8': '1'}

print(f'Project root : {BASE}')
print(f'Definition   : {DEFN_XLSX.relative_to(BASE)}')
print(f'Module       : {MOD}  (prefix {PREFIX}, fields {STUB}_*)')
print(f'Sampling     : {CELL_N} per year x contact reason x resolution cell (groups < {COLLAPSE_MIN} -> Other)')
print(f'Gate         : RUN_SAMPLE={RUN_SAMPLE}')

## 0b. Shared helpers

In [ ]:
def read_stage_output(path) -> pd.DataFrame:
    """Read a stage output, keeping the id column as a clean string."""
    path = Path(path)
    df = pd.read_excel(path) if path.suffix.lower().startswith('.xls') else pd.read_csv(path)
    df[ID_COL] = df[ID_COL].astype(str).str.strip()
    return df


def run_cmd(cmd, label):
    """Run a short command, echoing its output; raise on failure."""
    print(f'\n$ {" ".join(str(c) for c in cmd)}')
    r = subprocess.run(cmd, text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
    print(r.stdout or '', end='')
    print(r.stderr or '', end='')
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed (exit {r.returncode})')


def stratified_sample(df, by, n, seed=SEED, exclude=()) -> pd.DataFrame:
    """n rows spread evenly over the strata in `by`, topped up at random."""
    pool = df[~df[ID_COL].isin(set(exclude))]
    groups = pool.groupby(by, observed=True)
    per = max(1, math.ceil(n / groups.ngroups))
    parts = [g.sample(n=min(per, len(g)), random_state=seed) for _, g in groups]
    picked = pd.concat(parts)
    if len(picked) > n:
        picked = picked.sample(n=n, random_state=seed)
    elif len(picked) < n:
        rest = pool[~pool[ID_COL].isin(set(picked[ID_COL]))]
        picked = pd.concat([picked, rest.sample(n=min(n - len(picked), len(rest)),
                                                random_state=seed)])
    return picked.reset_index(drop=True)


print('Helpers defined.')

## 0c. Safety net — back up the shared config

`rca build build` rebuilds `generated/pipeline_config.json` from its input workbook only, so
whichever product is live right now gets dropped. It is backed up here first, with the command to
put it back.

In [ ]:
CONFIG_BACKUP.mkdir(parents=True, exist_ok=True)
_live = json.loads(CONFIG_JSON.read_text(encoding='utf-8'))
_live_modules = [m for m in _live['modules'] if m != MOD]

_backed_up = []
for _src in [CONFIG_JSON, BASE / 'generated' / 'build_manifest.json',
             *[TAXONOMY_DIR / f'{m}.json' for m in _live_modules]]:
    if _src.exists():
        _dst = CONFIG_BACKUP / _src.name
        if not _dst.exists():                 # never clobber an earlier backup
            shutil.copy2(_src, _dst)
        _backed_up.append(_src.name)
print(f'Backup dir: {CONFIG_BACKUP.relative_to(BASE)}  ({len(_backed_up)} files)')
print(f'Modules in the live config right now: {list(_live["modules"])}')

_DEFN_OF = {'sd_ac_': 'sd_2026_accessing_content_definition.xlsx',
            'sd_': 'sd_2026_pipeline_definition.xlsx',
            'csd_': 'csd_2026_pipeline_definition.xlsx',
            'regid_': 'reg_id_2026_pipeline_definition.xlsx',
            'leapspace': 'leapspace_2026_pipeline_definition.xlsx'}
_defn = next((v for k, v in _DEFN_OF.items()
              if any(m.startswith(k) for m in _live_modules)), None)
_restore = (f'rca build --input definitions/{_defn} --force-all'
            if _defn else '(no known definition workbook rebuilds these - see generated/)')

# Written once. On a second run the live config is already this notebook's own,
# and overwriting would lose the only record of what was there before.
_restore_txt = CONFIG_BACKUP / 'RESTORE.txt'
if not _restore_txt.exists():
    _restore_txt.write_text(
        'Modules live in generated/pipeline_config.json before notebook 17 ran:\n'
        f'    {", ".join(_live_modules) or "(none)"}\n\nTo put them back:\n\n    {_restore}\n',
        encoding='utf-8')
print('\n' + _restore_txt.read_text(encoding='utf-8'))

## 1. Load and filter the ticket list *(free)*

The xlsx is used for the filter and the metadata only. Its `Conversation` column is read once, to
count how many messages it shows per ticket, and then dropped.

Three metadata columns are renamed because their names start with prefixes that
`drop_enrichment_cols()` strips silently (`resolution_`, `contact_reason_`).

In [ ]:
_t0 = time.time()
xlsx = pd.read_excel(TICKETS_XLSX)
print(f'{TICKETS_XLSX.name}: {len(xlsx):,} rows x {len(xlsx.columns)} cols ({time.time() - _t0:.0f}s)')

xlsx[ID_COL] = xlsx[ID_COL].astype(str).str.strip()
_value_add = xlsx['IS_VALUE_ADD'].astype(str).str.strip().str.upper().eq('TRUE')
_both = xlsx['_merge'].astype(str).str.strip().eq('both')
print(f'IS_VALUE_ADD == TRUE : {_value_add.sum():,}')
print(f"_merge == 'both'     : {_both.sum():,}")

tickets = xlsx[_value_add & _both].copy()
print(f'Both                 : {len(tickets):,}')
assert tickets[ID_COL].is_unique, f'{ID_COL} is not unique after filtering'
assert len(tickets) == EXPECTED_TICKETS, (
    f'{len(tickets):,} tickets, expected {EXPECTED_TICKETS:,} - has the export changed?')

# Messages per ticket as the xlsx shows them, kept only to measure the duplication.
tickets['xlsx_messages'] = tickets[TEXT_COL].fillna('').str.count(r'(?m)^(?:Customer|Agent):')
tickets['xlsx_chars'] = tickets[TEXT_COL].fillna('').str.len()
XLSX_EXAMPLE = tickets[[ID_COL, TEXT_COL]].copy()     # for the before/after check in section 2

META_RENAME = {'resolution_code_lvl1_desc': 'rc_lvl1',
               'contact_reason_lvl1_desc': 'cr1_desc',
               'contact_reason_lvl2_desc': 'cr2_desc'}
META_KEEP = [ID_COL, 'INCIDENT_ID', 'Channel', 'Chat_Start_Date', 'incident_closed_datetime',
             'incident_subject', 'number_of_responses', 'agent_group_name', 'cr1_desc', 'cr2_desc',
             'customer_classification_type', 'rc_lvl1', 'status_desc',
             'incident_to_incident_relationship', 'xlsx_messages', 'xlsx_chars']
meta = tickets.rename(columns=META_RENAME)[META_KEEP].reset_index(drop=True)
del xlsx

print(f'\nChannel:\n{meta["Channel"].value_counts().to_string()}')
print(f'\nTop resolution codes:\n{meta["rc_lvl1"].value_counts().head(10).to_string()}')
print(f'\nAgent group:\n{meta["agent_group_name"].value_counts().to_string()}')

## 2. Rebuild the transcripts from the raw thread export *(free)*

Per entry: deduplicate, clean the HTML, cut the email signature and any quoted earlier email, and
collapse the automated templates to one line. Then each ticket becomes one transcript where every
turn carries a header:

```
[T5 | 2025-04-04 | +7d | Agent A2] Dear ..., the document below has been raised to the technical team ...
```

- `T5`: turn number. The model cites these, so an evidence quote can be found again.
- `+7d`: days since the previous turn, so waits are visible.
- `Agent A1`, `A2`, …: per-ticket aliases in order of first appearance. A change of agent is visible
  without exposing staff IDs.
- `System (automated)`: acknowledgement and "ticket still open" reminders.
- `Customer (auto-reply)`: out-of-office and other auto-responders. Not customer effort.

In [ ]:
_t0 = time.time()
raw = pd.read_csv(RAW_TSV, sep='\t', dtype=str, keep_default_na=False)
print(f'{RAW_TSV.name}: {len(raw):,} entries, {raw[ID_COL].nunique():,} incidents '
      f'({time.time() - _t0:.0f}s)')
print(f'Entry types: {raw["THREAD_ENTRY_TYPE"].value_counts().to_dict()}')
assert set(raw['THREAD_ENTRY_TYPE']) <= {*CUSTOMER_TYPES, STAFF_TYPE}, (
    'a new entry type has appeared - it may be the private notes this notebook assumes are absent')

raw[ID_COL] = raw[ID_COL].str.strip()
raw = raw[raw[ID_COL].isin(set(meta[ID_COL]))].copy()
_dup = raw.duplicated(['INCIDENT_ID', 'THREAD_ID', 'sequence'])
print(f'\nIn scope: {len(raw):,} entries; exact duplicates on (INCIDENT_ID, THREAD_ID, sequence): '
      f'{_dup.sum():,} ({_dup.mean():.1%}) - dropped')
raw = raw[~_dup].copy()

raw['dt'] = pd.to_datetime(raw['DATE_CREATED'], errors='coerce')
assert raw['dt'].notna().all(), 'unparseable DATE_CREATED'
raw['_thread'] = pd.to_numeric(raw['THREAD_ID'], errors='coerce')
raw['_seq'] = pd.to_numeric(raw['sequence'], errors='coerce')
raw = raw.sort_values([ID_COL, 'dt', '_thread', '_seq']).reset_index(drop=True)

_missing = sorted(set(meta[ID_COL]) - set(raw[ID_COL]))
print(f'Tickets with no thread entries at all: {len(_missing):,}'
      + (f'  e.g. {_missing[:5]}' if _missing else ''))
print(f'Longest raw entry: {raw["VERBATIM"].str.len().max():,} chars (source cap is 9,000)')

In [ ]:
_STYLE  = re.compile(r'(?is)<(style|script|head)[^>]*>.*?</\1>')
_BREAK  = re.compile(r'(?i)<br\s*/?>|</p>|</div>|</li>|</tr>|</h\d>')
_TAG    = re.compile(r'<[^>]+>')
_SPACES = re.compile(r'[ \t\r\f\v]+')
_NL     = re.compile(r' *\n[\n ]*')


def clean_html(s: str) -> str:
    s = _STYLE.sub(' ', s)
    s = _BREAK.sub('\n', s)
    s = _TAG.sub(' ', s)
    s = html.unescape(s).replace('\xa0', ' ').replace('\u200b', '').replace('\ufeff', '')
    s = _SPACES.sub(' ', s)
    return _NL.sub('\n', s).strip()


# A staff signature: the last closing phrase, when what follows it is short and
# names Elsevier / the service desk.
_CLOSING = re.compile(r'(?i)\b(kind regards|best regards|warm regards|warmest regards|'
                      r'regards|sincerely|best wishes|many thanks)\b')
_SIG_TAIL = re.compile(r'(?i)elsevier|customer service|support cent(er|re)|service desk')


def strip_signature(s: str) -> str:
    m = list(_CLOSING.finditer(s))
    if m:
        tail = s[m[-1].start():]
        if len(tail) < 600 and _SIG_TAIL.search(tail):
            return s[:m[-1].start()].rstrip(' ,\n')
    return s


# An earlier email quoted under a reply. Cut only when something precedes it, or
# a forwarded request would be emptied.
_QUOTED = re.compile(
    r'(?i)(?:^|\n|\s)(?:-{2,}\s*original message\s*-{2,}'
    r'|(?:from|de|von|van)\s*:\s[^\n]{1,250}?\s(?:sent|date|enviado|envoy[ée]|gesendet|verzonden)\s*:'
    r'|on\s[^\n]{0,80}?\d[^\n]{0,80}?\swrote:'
    # "<date> <sender> wrote:" in several languages - only when it ends the line,
    # so a customer writing "as I wrote: ..." keeps their message.
    r'|[^\n]{5,200}?\s(?:wrote|ha scritto|a [ée]crit|schrieb|escribi[óo]|escreveu|napisa[łl]|'
    r'написал\w*|schreef)\s*:[ \t]*(?:\n|$))')


def strip_quoted(s: str) -> str:
    m = _QUOTED.search(s)
    return s[:m.start()].rstrip() if m and m.start() > 40 else s


def cap_turn(s: str) -> str:
    if len(s) <= TURN_CAP:
        return s
    return s[:TURN_HEAD] + ' …[message shortened]… ' + s[-(TURN_CAP - TURN_HEAD - 25):]


# Automated staff templates -> one line. Matched on text: the same shared accounts
# also send real replies.
AUTO_TEMPLATES = [
    (re.compile(r'(?i)(will be|has been) assigned to a Content Service Desk support specialist'),
     'Automated acknowledgement: ticket received and queued for a Content Service Desk specialist.'),
    (re.compile(r'(?i)confirming that it remains open'),
     'Automated reminder: ticket still open, "our technical team is diligently working" on it.'),
    (re.compile(r'(?i)thank you for taking the time to respond to our survey'),
     'Automated survey acknowledgement.'),
]
_OOO = re.compile(r'(?i)out of (the )?office|automatic reply|auto-?reply|autoreply|'
                  r'away from (the |my )?(office|desk)|on (annual |sick )?leave|currently away|'
                  r'limited access to (my )?e-?mail|abwesend|absence du bureau|fuera de la oficina')


def classify_entry(entry_type: str, text: str) -> tuple:
    """-> (kind, text). kind: customer | customer_auto | agent | system."""
    if entry_type == STAFF_TYPE:
        for rx, label in AUTO_TEMPLATES:
            if rx.search(text):
                return 'system', label
        return 'agent', text
    if _OOO.search(text[:400]) and len(text) < 1_500:
        return 'customer_auto', text
    return 'customer', text


_t0 = time.time()
raw['text'] = raw['VERBATIM'].map(clean_html)
_staff = raw['THREAD_ENTRY_TYPE'].eq(STAFF_TYPE)
raw.loc[_staff, 'text'] = raw.loc[_staff, 'text'].map(strip_signature)
raw['text'] = raw['text'].map(strip_quoted)
_kt = [classify_entry(e, t) for e, t in zip(raw['THREAD_ENTRY_TYPE'], raw['text'])]
raw['kind'] = [k for k, _ in _kt]
raw['text'] = [cap_turn(t) for _, t in _kt]
turns = raw[raw['text'].str.strip().ne('')].copy()
print(f'Cleaned {len(raw):,} entries in {time.time() - _t0:.0f}s; '
      f'{len(raw) - len(turns):,} empty after cleaning, dropped')
print(f'Turn kinds: {turns["kind"].value_counts().to_dict()}')
print(f'Characters: raw HTML {raw["VERBATIM"].str.len().sum() / 1e6:,.0f}M -> '
      f'clean {turns["text"].str.len().sum() / 1e6:,.0f}M')

In [ ]:
# ---- Per-ticket aliases, turn numbers, gaps -----------------------------
turns['turn_no'] = turns.groupby(ID_COL).cumcount() + 1
_agent = turns['kind'].eq('agent')
_first_seen = turns.loc[_agent, [ID_COL, 'AGENT_ID']].drop_duplicates()
_first_seen['agent_alias'] = 'A' + (_first_seen.groupby(ID_COL).cumcount() + 1).astype(str)
turns = turns.merge(_first_seen, on=[ID_COL, 'AGENT_ID'], how='left')   # keeps turn order
turns.loc[~turns['kind'].eq('agent'), 'agent_alias'] = ''
turns['gap_days'] = turns.groupby(ID_COL)['dt'].diff().dt.total_seconds() / 86_400

_SPEAKER = {'customer': 'Customer', 'customer_auto': 'Customer (auto-reply)',
            'system': 'System (automated)'}


def header(r) -> str:
    who = f'Agent {r.agent_alias}' if r.kind == 'agent' else _SPEAKER[r.kind]
    gap = '' if pd.isna(r.gap_days) else f' | +{int(r.gap_days)}d'
    return f'[T{r.turn_no} | {r.dt:%Y-%m-%d}{gap} | {who}]'


turns['line'] = [f'{header(r)} {r.text}' for r in turns.itertuples()]

TRUNC_MARKER = '\n\n…[MIDDLE OF CONVERSATION OMITTED FOR LENGTH]…\n\n'
assert TRUNC_HEAD + len(TRUNC_MARKER) + TRUNC_TAIL <= TRUNC_CAP


def cap(text: str) -> str:
    return text if len(text) <= TRUNC_CAP else text[:TRUNC_HEAD] + TRUNC_MARKER + text[-TRUNC_TAIL:]


conv = turns.groupby(ID_COL, sort=False)['line'].agg('\n'.join).rename('transcript_full')
transcripts = meta[[ID_COL]].merge(conv, left_on=ID_COL, right_index=True, how='left')
transcripts['has_transcript'] = transcripts['transcript_full'].notna()
transcripts['transcript_full'] = transcripts['transcript_full'].fillna('')
transcripts['transcript_chars'] = transcripts['transcript_full'].str.len()
transcripts['was_truncated'] = transcripts['transcript_chars'] > TRUNC_CAP
transcripts[TEXT_COL] = transcripts['transcript_full'].map(cap)
assert transcripts[TEXT_COL].str.len().max() <= TRUNC_CAP

print(f'Tickets with a transcript : {transcripts["has_transcript"].sum():,} of {len(transcripts):,}')
print(f'Transcript chars (median / p90 / max): '
      f'{transcripts["transcript_chars"].median():,.0f} / '
      f'{transcripts["transcript_chars"].quantile(.9):,.0f} / {transcripts["transcript_chars"].max():,}')
print(f'Truncated to {TRUNC_CAP:,} chars: {transcripts["was_truncated"].sum():,}')

# ---- Duplication check: turns per ticket, xlsx vs rebuilt ---------------
_n_turns = turns.groupby(ID_COL).size()
_cmp = meta[[ID_COL, 'xlsx_messages']].assign(rebuilt=meta[ID_COL].map(_n_turns).fillna(0))
print(f'\nMessages shown, all tickets: xlsx {_cmp["xlsx_messages"].sum():,} -> rebuilt '
      f'{int(_cmp["rebuilt"].sum()):,} ({1 - _cmp["rebuilt"].sum() / _cmp["xlsx_messages"].sum():.1%} fewer)')
print(f'Tickets where the xlsx shows more messages than exist: '
      f'{(_cmp["xlsx_messages"] > _cmp["rebuilt"]).sum():,}')

transcripts[[ID_COL, 'transcript_full']].to_csv(TRANSCRIPTS_CSV, index=False, encoding='utf-8')
print(f'\nWrote {TRANSCRIPTS_CSV.relative_to(BASE)}')

In [ ]:
# ---- Before / after, one ticket where the duplication is visible --------
_ex_id = _cmp.loc[(_cmp['xlsx_messages'] >= 2 * _cmp['rebuilt']) & (_cmp['rebuilt'].between(4, 8)),
                  ID_COL].iloc[0]
_lines = XLSX_EXAMPLE.loc[XLSX_EXAMPLE[ID_COL] == _ex_id, TEXT_COL].iloc[0].split('\n')
print(f'=== {_ex_id}: xlsx Conversation - {len(_lines)} lines, each cut to 110 chars ===')
for _l in _lines:
    print(f'  {_l[:110]}')
print(f'\n=== {_ex_id}: rebuilt transcript ===')
print(transcripts.loc[transcripts[ID_COL] == _ex_id, TEXT_COL].iloc[0][:4_000])
del XLSX_EXAMPLE

## 3. Rule-based features, every ticket *(free)*

Two independent readings that cost nothing and cover all 63.8k tickets:

**`obs_*`: the shape of the interaction.** Same logic as `scripts/build_effort_features.py`,
run on the deduplicated turns, with automated messages and auto-replies kept out of the counts.

| column | meaning |
|---|---|
| `obs_customer_turns`, `obs_agent_turns`, `obs_system_turns`, `obs_customer_autoreplies` | turn counts by kind |
| `obs_distinct_agents`, `obs_agent_changes` | human agents on the ticket; times the replying agent changed |
| `obs_span_days` | first to last turn |
| `obs_first_response_h` | first customer turn → first human agent reply |
| `obs_max_wait_days` | longest wait from a customer turn to the next human agent reply |
| `obs_customer_followups` | customer turns after the first human reply |
| `obs_repeat_contact`, `obs_contact_ticket_count`, `obs_prior_tickets_90d` | same `CONTACT_ID` on other tickets **in this population**; NA when the contact is unknown |
| `obs_contact_is_bulk` | the contact has 50+ tickets: an account manager, CSM desk or proxy, not one researcher |

**`rx_*`: a keyword pass for escalation, returns, notes and chasing.** This is a cross-check for the
LLM and a stratum for sampling, not the answer. It cannot tell "I have raised this to the technical
team" from "I will raise this if needed".

In [ ]:
# ---- Regexes -------------------------------------------------------------
_TEAM = (r'(?:technical|tech|second[- ]?line|2nd[- ]?line|2LS|L2|level[- ]?2|content|indexing|'
         r'production|data|profile|author feedback|specialist|expert|relevant|concerned|'
         r'appropriate|responsible|dedicated|internal|back[- ]?end|supplier|vendor)'
         r'(?:\s+support)?\s+(?:team|department|group|colleagues|specialists?)')
ESC_RX = re.compile(
    r'(?i)\b(?:rais|forward|escalat|refer|endors|pass|submit|report|transfer|assign|rout|send|sent|'
    r'shar|flag|relay|contact|reach(?:ed)? out to|coordinat|collaborat|consult|check(?:ing)? with|'
    r'inform)\w*\b[^.\n]{0,80}?' + _TEAM
    + r'|\bsent to (?:the )?(?:supplier|vendor)\b|\b(?:second[- ]line|2LS)\b'
    # Agreed with the business: this template means the case was sent for processing.
    r'|\binitiated the (?:process|request|correction)\b|\bprocess to (?:amend|add|correct)\b[^.\n]{0,60}\binitiated\b')
RETURN_RX = re.compile(
    r'(?i)\b(?:' + _TEAM + r'|they|the team)\s+(?:has|have|had)?\s*(?:now\s+)?'
    r'(?:advised|confirmed|informed|responded|replied|requested|asked|updated|come back|got back|'
    r'reverted|stated|mentioned|clarified|explained|indicated|reviewed|completed|declined|rejected)'
    r'|\breceived (?:the |an? )?(?:update|feedback|response|reply|confirmation) (?:from|of)\b'
    r'|\b(?:team|they)\s+(?:has|have)\s+(?:now\s+)?(?:been able|managed|successfully)\b'
    r'|\bas (?:per|advised by) (?:our|the) ' + _TEAM
    + r'|\baccording to (?:our|the) ' + _TEAM)
NOTE_RX = re.compile(
    r'(?i)\b(?:private|internal)\s+note\b|\b1LS\b|\b2LS\b|\bL1\b(?=.{0,40}\bL2\b)'
    r'|^dear (?:team|all|2ls|support team)\b|\bspecial handling\b|\bplease (?:check|assist|advise|handle|action)\b'
    r'(?=[^.]{0,60}\b(?:team|ticket|task|case)\b)')
CHASE_RX = re.compile(
    r'(?i)\bany (?:update|news|progress|feedback)\b|\bstill (?:not|no|waiting|missing|shows?|showing|incorrect|wrong)\b'
    r'|\bfollow(?:ing)?[- ]up\b|\breminder\b|\bhow long\b|\bwhen will\b|\bno (?:response|reply|answer)\b'
    r'|\bhaven\'?t (?:heard|received)\b|\bnot (?:yet )?(?:been )?(?:resolved|fixed|added|updated|corrected|indexed)\b'
    r'|\bas (?:i|we) (?:mentioned|said|explained)\b|\balready (?:sent|provided|explained)\b|\bre-?send(?:ing)?\b')

# ---- Self-test on phrasings seen in the data ------------------------------
for _s in ['I have raised this to our technical team for checking.',
           "I'll forward it to our relevant team for better assistance.",
           'Please be advised that I have forwarded your request to our Second Line Support Team',
           'The document below has been raised to the technical team for further evaluation.',
           'Following your enquiry, I have initiated the process to add your paper to our database.',
           'I can confirm that the process to amend the data in our database has been initiated.']:
    assert ESC_RX.search(_s), f'ESC_RX misses: {_s}'
for _s in ['They have informed me that this paper is reflected in Scopus via Medline.',
           'Our technical team has confirmed that the correction is complete.']:
    assert RETURN_RX.search(_s), f'RETURN_RX misses: {_s}'
assert RETURN_RX.search('Please be informed that, I have now received the response of our technical team.')
for _s in ['I left a priority private note for our second-line support team',
           'there should be a dedicated group looking into this on both 1LS and 2LS']:
    assert NOTE_RX.search(_s), f'NOTE_RX misses: {_s}'
assert CHASE_RX.search('Is there any update on my request?')
for _s, _want in [('Thanks, the DOI of this article is still not fixed.\nIl giorno gio 3 apr 2025 alle ore 07:50 Scopus Support ha scritto:',
                   'Thanks, the DOI of this article is still not fixed.'),
                  ('Please see my earlier mail. As I wrote: the DOI is wrong and must be fixed.',
                   'Please see my earlier mail. As I wrote: the DOI is wrong and must be fixed.')]:
    assert strip_quoted(_s) == _want, f'strip_quoted({_s[:40]!r}...) -> {strip_quoted(_s)!r}'
print('Regex self-tests pass.')

In [ ]:
_t0 = time.time()
t = turns[[ID_COL, 'turn_no', 'dt', 'kind', 'AGENT_ID', 'CONTACT_ID', 'text']].copy()
is_cust, is_agent = t['kind'].eq('customer'), t['kind'].eq('agent')
g = t.groupby(ID_COL)

f = pd.DataFrame({'_t_first': g['dt'].min(), '_t_last': g['dt'].max()})
for _col, _kind in [('obs_customer_turns', 'customer'), ('obs_agent_turns', 'agent'),
                    ('obs_system_turns', 'system'), ('obs_customer_autoreplies', 'customer_auto')]:
    f[_col] = t[t['kind'].eq(_kind)].groupby(ID_COL).size().reindex(f.index).fillna(0).astype(int)
f['obs_distinct_agents'] = t[is_agent].groupby(ID_COL)['AGENT_ID'].nunique().reindex(f.index).fillna(0).astype(int)
_ag = t[is_agent]
f['obs_agent_changes'] = (_ag['AGENT_ID'].ne(_ag.groupby(ID_COL)['AGENT_ID'].shift())
                          & _ag.groupby(ID_COL).cumcount().gt(0)).groupby(_ag[ID_COL]).sum() \
                          .reindex(f.index).fillna(0).astype(int)
f['obs_span_days'] = (f['_t_last'] - f['_t_first']).dt.total_seconds() / 86_400

# First human reply after the first customer turn
_c0 = t[is_cust].groupby(ID_COL)['dt'].min().rename('_c0')
_a = t[is_agent][[ID_COL, 'dt']].join(_c0, on=ID_COL)
_first_reply = _a[_a['dt'] >= _a['_c0']].groupby(ID_COL)['dt'].min()
f['obs_first_response_h'] = ((_first_reply - _c0).dt.total_seconds() / 3_600).reindex(f.index)
f['obs_customer_followups'] = (t[is_cust].join(_first_reply.rename('_r1'), on=ID_COL)
                               .pipe(lambda d: d[d['dt'] > d['_r1']]).groupby(ID_COL).size()
                               .reindex(f.index).fillna(0).astype(int))

# Longest wait: each customer turn -> next human agent turn (next_agent_dt via backward fill)
_na = t['dt'].where(is_agent)
t['_next_agent_dt'] = _na.groupby(t[ID_COL]).bfill()
_w = t[is_cust]
f['obs_max_wait_days'] = ((_w['_next_agent_dt'] - _w['dt']).dt.total_seconds() / 86_400) \
    .groupby(_w[ID_COL]).max().reindex(f.index)

# Repeat contact within this population
_contact = t[t['kind'].isin(['customer', 'customer_auto']) & t['CONTACT_ID'].ne('')] \
    .groupby(ID_COL)['CONTACT_ID'].first()
f['obs_contact_id'] = _contact.reindex(f.index)
_known = f['obs_contact_id'].notna()
f['obs_contact_ticket_count'] = f['obs_contact_id'].map(f.loc[_known].groupby('obs_contact_id').size())
f['obs_repeat_contact'] = pd.Series(np.where(_known, f['obs_contact_ticket_count'] > 1, None),
                                    index=f.index, dtype='boolean')
# A few CONTACT_IDs sit on hundreds of tickets - account managers, CSM desks, proxies -
# so repeat contact means little for them. Flagged rather than dropped.
f['obs_contact_is_bulk'] = f['obs_contact_ticket_count'] > 50
_k = f.loc[_known, ['obs_contact_id', '_t_first']].sort_values(['obs_contact_id', '_t_first'])
_prior = {}
for _, grp in _k.groupby('obs_contact_id', sort=False):
    times = grp['_t_first'].to_numpy()
    lo = np.searchsorted(times, times - np.timedelta64(90, 'D'), side='left')
    _prior.update(zip(grp.index, np.arange(len(times)) - lo))
f['obs_prior_tickets_90d'] = pd.Series(_prior, dtype='Int64').reindex(f.index)

# ---- rx_* -----------------------------------------------------------------
t['_esc'] = is_agent & t['text'].str.contains(ESC_RX)
t['_ret'] = is_agent & t['text'].str.contains(RETURN_RX)
t['_note'] = is_agent & t['text'].str.contains(NOTE_RX)
t['_chase'] = is_cust & t['text'].str.contains(CHASE_RX)
g = t.groupby(ID_COL)
f['rx_escalation_turns'] = g['_esc'].sum().reindex(f.index).astype(int)
f['rx_return_turns'] = g['_ret'].sum().reindex(f.index).astype(int)
f['rx_internal_note_turns'] = g['_note'].sum().reindex(f.index).astype(int)
f['rx_customer_chasers'] = g['_chase'].sum().reindex(f.index).astype(int)
f['rx_first_escalation_turn'] = t[t['_esc']].groupby(ID_COL)['turn_no'].min().reindex(f.index)
f['rx_internal_note_example'] = t[t['_note']].groupby(ID_COL)['text'].first().str[:300].reindex(f.index)
f['rx_escalated'] = f['rx_escalation_turns'] > 0


def _episodes(flags) -> int:
    """Escalation episodes: an escalation turn that follows a return (or opens the ticket's sequence)."""
    n, armed = 0, True
    for esc, ret in flags:
        if esc and armed:
            n, armed = n + 1, False
        if ret:
            armed = True
    return n


_ev = t[t['_esc'] | t['_ret']]
f['rx_escalation_episodes'] = pd.Series(
    {k: _episodes(zip(v['_esc'], v['_ret'])) for k, v in _ev.groupby(ID_COL)}, dtype='Int64') \
    .reindex(f.index).fillna(0).astype(int)

features = meta[[ID_COL]].merge(f.drop(columns=['_t_first', '_t_last']).reset_index(),
                                on=ID_COL, how='left', validate='1:1')
features['rx_escalated'] = features['rx_escalated'].fillna(False).astype(bool)   # no transcript -> no wording
features.to_csv(FEATURES_CSV, index=False, encoding='utf-8')
print(f'{len(features):,} tickets x {features.shape[1] - 1} features in {time.time() - _t0:.0f}s '
      f'-> {FEATURES_CSV.relative_to(BASE)}')
print(f'Contact resolvable for {_known.mean():.1%}; repeat rate among those '
      f'{f.loc[_known, "obs_repeat_contact"].mean():.1%}\n')
print(features.describe(percentiles=[.5, .9]).T[['mean', '50%', '90%', 'max']].round(2).to_string())

In [ ]:
print(f'rx_escalated: {features["rx_escalated"].mean():.1%} of tickets carry explicit '
      f'hand-off wording from a human agent')
print('\nrx_escalation_episodes:')
print(features['rx_escalation_episodes'].clip(upper=3).value_counts().sort_index()
      .rename(index={3: '3+'}).to_string())
print(f'\nTickets with a note-like staff entry: {(features["rx_internal_note_turns"] > 0).sum():,}')
print(f'Tickets with no hand-off wording but at least one "technical team is working" reminder: '
      f'{((~features["rx_escalated"]) & (features["obs_system_turns"] > 0)).sum():,} '
      f'(system reminders alone are not counted as escalation)')

print('\n--- 8 note-like staff entries ---')
for _s in features['rx_internal_note_example'].dropna().sample(8, random_state=SEED):
    print(f'  {_s[:250]!r}')
print('\n--- rx_escalated by resolution code (top 12 by volume) ---')
_by = features.merge(meta[[ID_COL, 'rc_lvl1']], on=ID_COL)
_top = _by['rc_lvl1'].value_counts().head(12).index
print(_by[_by['rc_lvl1'].isin(_top)].groupby('rc_lvl1')['rx_escalated']
      .agg(['size', 'mean']).sort_values('size', ascending=False)
      .rename(columns={'size': 'tickets', 'mean': 'rx_escalated'}).round(3).to_string())

## 3b. Sampling design *(free)*

The LLM reads a sample, not all 63,824 tickets. The sample is built so results can be reported by
**year × contact reason × resolution code**.

- **Strata.**
  - Year of `Chat_Start_Date`. The 16 tickets from Nov 2024 are folded into 2025.
  - Contact reason level 1 (`cr1_desc`).
  - Resolution code (`rc_lvl1`).
  Contact reasons and resolution codes with fewer than `COLLAPSE_MIN` tickets in the population are
  grouped into "Other contact reason" / "Other resolution". Otherwise dozens of 1–10-ticket cells
  would each be read in full and still be too small to say anything.
- **Allocation.** `CELL_N` tickets per cell, or the whole cell if it is smaller. The big cells
  (e.g. 2026 × Add Missing Document × Sent to Supplier, ~13k tickets) are heavily under-sampled on
  purpose. The small ones are read in full. Equal allocation is what makes each cell reportable; a
  proportional sample would leave most cells with a handful of tickets.
- **Weights.** Every sampled ticket carries `weight = cell population / cell sample`. Every
  population figure in section 6 is weighted, so the over-sampled small cells do not distort the
  totals. The weights sum to exactly 63,824.
- **Reportable.** A cell is flagged `reportable` when it has ≥ `COLLAPSE_MIN` tickets in the
  population and ≥ `REPORT_MIN_SAMPLED` in the sample. At 50 sampled, a proportion is accurate to about
  ±14 points at worst (95%).

In [ ]:
frame = meta[[ID_COL, 'Chat_Start_Date', 'cr1_desc', 'rc_lvl1']].copy()
frame['year'] = pd.to_datetime(frame['Chat_Start_Date']).dt.year.clip(lower=2025).astype(str)


def _group_rare(s: pd.Series, other: str) -> pd.Series:
    s = s.fillna('(blank)').astype(str)
    return s.where(s.map(s.value_counts()) >= COLLAPSE_MIN, other)


frame['cr1_rep'] = _group_rare(frame['cr1_desc'], 'Other contact reason')
frame['rc_rep'] = _group_rare(frame['rc_lvl1'], 'Other resolution')
frame['cell'] = frame['year'] + ' | ' + frame['cr1_rep'] + ' | ' + frame['rc_rep']

# Only tickets with a transcript can be read; the population counts come from the same set.
frame = frame[frame[ID_COL].isin(set(transcripts.loc[transcripts['has_transcript'], ID_COL]))]
N_cell = frame.groupby('cell').size()

sample = pd.concat([g.sample(n=min(len(g), CELL_N), random_state=SEED)
                    for _, g in frame.groupby('cell', sort=True)])
n_cell = sample.groupby('cell').size()
sample['weight'] = sample['cell'].map(N_cell / n_cell)

alloc = (pd.DataFrame({'population': N_cell, 'sampled': n_cell})
         .join(frame.drop_duplicates('cell').set_index('cell')[['year', 'cr1_rep', 'rc_rep']]))
alloc['weight'] = (alloc['population'] / alloc['sampled']).round(2)
alloc['reportable'] = (alloc['population'] >= COLLAPSE_MIN) & (alloc['sampled'] >= REPORT_MIN_SAMPLED)
alloc = alloc.reset_index()[['year', 'cr1_rep', 'rc_rep', 'population', 'sampled', 'weight', 'reportable', 'cell']] \
    .sort_values(['year', 'population'], ascending=[True, False]).reset_index(drop=True)

# ---- Checks ---------------------------------------------------------------
assert sample[ID_COL].is_unique, 'a ticket was drawn twice'
assert abs(sample['weight'].sum() - len(frame)) < 1e-6, 'weights do not sum to the population'
assert set(sample['year']) == set(frame['year']), 'a year is missing from the sample'
assert set(sample['cr1_rep']) == set(frame['cr1_rep']), 'a contact reason is missing from the sample'
assert set(sample['rc_rep']) == set(frame['rc_rep']), 'a resolution code is missing from the sample'

_rep = alloc[alloc['reportable']]
print(f'Population {len(frame):,} tickets in {len(alloc)} cells '
      f'({frame["cr1_rep"].nunique()} contact reasons x {frame["rc_rep"].nunique()} resolution codes x '
      f'{frame["year"].nunique()} years, rare values grouped)')
print(f'Sample     {len(sample):,} tickets  (~${len(sample) * COST_PER_TICKET:,.0f}, '
      f'~{len(sample) / TICKETS_PER_SEC / 3600:.1f} h)')
print(f'Reportable cells: {len(_rep)} of {len(alloc)}, covering '
      f'{_rep["population"].sum() / len(frame):.1%} of the population')
print(f'\nSampled per year:\n{sample["year"].value_counts().sort_index().to_string()}')
print(f'\nSampled per contact reason:\n{sample["cr1_rep"].value_counts().to_string()}')
print(f'\nSampled per resolution code (top 15):\n{sample["rc_rep"].value_counts().head(15).to_string()}')
print('\nLargest cells (most down-weighted):')
print(alloc.head(10).drop(columns='cell').to_string(index=False))

sample[[ID_COL, 'cell', 'year', 'cr1_rep', 'rc_rep', 'weight']].to_csv(SAMPLE_FRAME, index=False, encoding='utf-8')
print(f'\nWrote {SAMPLE_FRAME.relative_to(BASE)}')

## 4. The LLM module: definition workbook and build *(free)*

One module, `s2ls_escalation_effort`, one prompt, every judgement at once. The model reads the
whole transcript once rather than twice.

**The escalation journey taxonomy (`SLJ`)** gives each ticket one code for its overall shape. It
is also the module's sentinel field for checkpointing.

**The evidence pattern.** Every judgement `s2_X` has `s2_X_evidence` (a verbatim quote) and
`s2_X_justification` (why, in ≤25 words, naming the turn). The pipeline's `validate_evidence_fields()`
checks every `*_evidence` column against the transcript, adds a `*_grounded` flag and blanks
fabricated quotes. The hand-off events and friction points are lists of objects, and each item
carries its own evidence.

`rca build` assigns taxonomy codes by row order: **append, never insert**, or the codes in
earlier outputs shift.

In [ ]:
# ---- Shared prompt language ---------------------------------------------
# Kept short on purpose: this prompt is sent once per ticket (batch_size 1) and
# was ~80% of the input tokens before it was trimmed (4.2k -> 3.0k tokens).
G_CONTEXT = (
    'Scopus Content support: 1LS agents talk to the customer; 2LS is the team behind them (agents say '
    '"technical team", "second-line", "Content team", "relevant team"), and beyond it the content suppliers.')
G_FORMAT = (
    'Each turn starts [T<n> | date | +days since previous turn | speaker]. Speakers: Customer; Customer '
    '(auto-reply) = auto-responder, never effort; Agent A1, A2... = human 1LS agents (new number = new person); '
    'System (automated) = template acknowledgements/"still open" reminders.')
G_ESCALATION = (
    'Escalation = a human agent turn saying the case was passed on or is waiting on 2LS/another internal '
    'team/the supplier (raised, forwarded, sent, contacted, escalated...), OR "I have initiated the process to '
    'add/amend/correct..." (this template means it was sent for processing), OR an agent relaying what the '
    'team said or did ("as per the technical team...", "they informed me..."). Agent raised it and later '
    'reports it done = resolved by 2LS, not by 1LS. NOT escalation: System turns alone; "under review" with '
    'nothing else; offering to escalate; redirecting the customer to the publisher or another service. '
    'Journey code, escalated, events and counts must agree.')
G_NOTES = (
    'Internal note = a staff turn written for colleagues, not the customer (to a team/colleague, mentions '
    '1LS/2LS/L1/L2, handling instructions, internal task numbers, "special handling procedure") - even when it '
    'opens "Dear <customer name>", e.g. "This should be tracked closely... a dedicated group looking into this '
    'on both 1LS and 2LS" or "a task has been assigned to 2LS, Task: 251016-008451". Also an agent telling the '
    'customer they left a private/internal note for 2LS.')
G_EVIDENCE = (
    'Every *_evidence = VERBATIM quote from the transcript, max 30 words, no [T..] header; empty string if the '
    'value is negative and there is nothing to quote. Every *_justification = max 25 words, names the turns '
    '(e.g. "T4, T9") and explains the judgement. Both refer to the field just before them.')

# ---- The escalation journey taxonomy -------------------------------------
# ORDER IS LOAD-BEARING: SLJ1.1 is the first leaf here. Append, never insert.
JOURNEY = {
    'Not escalated': {
        'Resolved at 1LS': '1LS answered or completed it with no other team involved.',
        'Rejected or out of scope at 1LS': '1LS itself declined it (policy, coverage, invalid, already correct).',
        'Closed for no customer response': '1LS asked the customer for information that never came.',
        'Redirected outside Scopus support': 'Customer sent to the publisher, another service or a self-service tool.',
    },
    'Escalated once, no back-and-forth': {
        'Escalated and resolved by 2LS': 'One hand-off; 2LS did the work or answered; relayed to the customer.',
        'Escalated and rejected by 2LS': (
            'One hand-off; 2LS declined or found Scopus already matches the publisher (customer told to ask '
            'the publisher).'),
        'Escalated, outcome not visible': 'One hand-off; transcript ends without an outcome.',
    },
    'Escalated with back-and-forth': {
        '2LS needed more information from the customer': (
            '2LS asked (via 1LS) for details or proof; customer replied; case went back to 2LS.'),
        'Re-escalated after customer pushback or reopening': (
            'Outcome came back, customer disputed or chased, 1LS sent it to 2LS again. Preferred when several '
            'patterns apply.'),
        'Multiple hand-offs between teams': 'Passed across several teams, or repeatedly re-raised with 2LS.',
    },
    'Unclear': {
        'Cannot tell from the transcript': 'Too short, only automated turns, or ambiguous.',
    },
}

FRICTION = {
    'repeated_information_request': 'asked for the same or more information more than once',
    're_explaining_issue': 'had to restate the problem, often to a new agent',
    'customer_chasing': 'had to follow up for status',
    'long_wait_no_update': 'weeks with no substantive update, or only automated reminders',
    'bounced_between_teams': 'passed between agents, teams or tiers',
    'unclear_or_generic_answer': 'vague, templated, contradictory or off-point replies',
    'request_rejected_or_policy': 'declined for policy, coverage or rules',
    'redirected_elsewhere': 'told to go to the publisher, another team or a form',
    'incorrect_or_incomplete_fix': 'correction wrong or partial, customer reported it',
    'unresolved_at_close': 'ends with the issue not resolved',
    'language_or_tone': 'language barrier or tone the customer objected to',
}
EVENT_TYPES = ['escalated', 're_escalated', 'chased_2ls', 'info_request_from_2ls',
               'answer_from_2ls', 'completed_by_2ls', 'rejected_by_2ls']
print(f'{sum(len(v) for v in JOURNEY.values())} journey leaves, {len(FRICTION)} friction categories, '
      f'{len(EVENT_TYPES)} event types')

In [ ]:
MOD_DESC = (
    'From one Scopus Content support transcript decide whether it was really escalated 1LS -> 2LS, how often '
    'it went back and forth, whether internal notes exist, and the customer effort - with evidence for each. '
    f'{G_CONTEXT} {G_FORMAT} {G_ESCALATION} {G_NOTES} {G_EVIDENCE} Read the whole transcript first.')

_friction_list = '; '.join(f"'{k}' = {v}" for k, v in FRICTION.items())
_events_list = ', '.join(EVENT_TYPES)


def F(name, ftype, desc, instr, explode_keys=''):
    return {'module_name': MOD, 'field_name': f'{STUB}_{name}', 'field_type': ftype,
            'description': desc, 'prompt_instruction': instr, 'in_output': 'TRUE',
            'explode_list': 'TRUE' if explode_keys else 'FALSE',
            'validate_enum': 'TRUE', 'notes': None, 'explode_keys': explode_keys or None}


def EJ(name, what):
    """The evidence + justification pair after every judgement; the rules live once, in G_EVIDENCE.
    (A blank instruction is not an option: build_pipeline reads it back as the string 'nan'.)"""
    # A one-word "Evidence." instruction was tried: the model paraphrased, and only 35% of
    # effort quotes were found in the transcript. Each field needs its own "copy exactly".
    return [F(f'{name}_evidence', 'str', f'Verbatim quote supporting {what}',
              f'Quote for {what}, copied EXACTLY from one turn (no paraphrase), max 30 words.'),
            F(f'{name}_justification', 'str', f'Why {what} was chosen',
              f'Why {what}: max 25 words, naming the turns.')]


fields_rows = [
    # ---- Escalation --------------------------------------------------------
    F('journey_code', 'code', 'Escalation journey leaf code',
      f'One leaf code (e.g. {PREFIX}2.1) for the overall journey.'),
    F('journey_justification', 'str', 'Why this journey code',
      'Max 25 words, naming turns: why this code and not the nearest one.'),
    F('escalated', 'enum:yes,no,unclear', 'Was the ticket escalated to 2LS',
      'yes = handed off at least once (see Escalation); no = handled, declined or redirected by 1LS alone.'),
    *EJ('escalated', 'the escalated value'),
    F('escalation_target', 'enum:technical_team_2ls,content_supplier,other_internal_team,none',
      'Who the case was handed to',
      'First team handed to: technical_team_2ls (technical/second-line/Content team, or process initiated); '
      'content_supplier (supplier named); other_internal_team; none.'),
    F('handoff_events', 'list', 'Ordered hand-off events between 1LS and 2LS',
      'Every hand-off event in order: {turn: "T5", direction: to_2ls|from_2ls, event, evidence (verbatim, '
      f'max 30 words)}}. event: {_events_list}. re_escalated = any hand-off after the first; chased_2ls = 1LS '
      'only asks 2LS for an update (to_2ls). EVERY agent turn relaying the team ("as per the technical team", '
      '"the team confirmed", "I received the response of our technical team") is a from_2ls event - a request '
      'for details attributed to the team is info_request_from_2ls. System turns are never events. [] if not '
      'escalated.',
      explode_keys='turn,direction,event,evidence'),
    F('escalation_count', 'int', 'Number of hand-offs 1LS -> 2LS',
      'Count of escalated + re_escalated events (1 if only a relayed 2LS answer proves it).'),
    F('return_count', 'int', 'Number of returns 2LS -> 1LS', 'Count of from_2ls events.'),
    F('handoff_justification', 'str', 'How the counts were derived', 'Max 25 words, naming turns: the counts.'),
    F('not_escalated_reason',
      'enum:resolved_at_1ls,rejected_at_1ls,customer_no_response,redirected_elsewhere,na',
      'Why the ticket did not reach 2LS', 'Why not escalated; na if escalated.'),
    F('internal_note_type', 'enum:none,note_in_thread,note_referenced,both', 'Internal notes found',
      'note_in_thread = a staff turn is an internal note; note_referenced = agent says a private/internal note '
      'was left for 2LS.'),
    *EJ('internal_note', 'the internal note finding'),
    F('internal_note_summary', 'str', 'What the note says',
      'Max 30 words: what the note says about escalation, urgency or status; else empty.'),
    # ---- Customer effort ---------------------------------------------------
    F('effort_score', 'enum:very_low,low,medium,high,very_high', 'Overall customer effort',
      'very_low = one exchange, clear answer; low = minor clarification; medium = several pieces of '
      'information, some back-and-forth or a wait; high = weeks waiting on 2LS, repeating, chasing or being '
      'passed around; very_high = many rounds, months, or unresolved.'),
    *EJ('effort', 'the effort score'),
    F('friction_points', 'list', 'Friction points with evidence',
      f'One object per evidenced friction: {{category, evidence, justification}}. category, exact label: '
      f'{_friction_list}. [] if none.',
      explode_keys='category,evidence,justification'),
    F('customer_chase_count', 'int', 'Times the customer chased',
      'Customer turns (not auto-replies) chasing for status or repeating an ignored request.'),
    *EJ('customer_chase', 'the chase count'),
    F('customer_repeat_info', 'bool', 'Customer had to repeat information',
      'true if the customer had to resend information or re-explain.'),
    *EJ('customer_repeat_info', 'the repeat-information value'),
    F('resolution_outcome', 'enum:resolved,partially_resolved,rejected,unresolved,unclear',
      'Outcome for the customer', 'Outcome for the customer at the end.'),
    *EJ('resolution_outcome', 'the outcome'),
    F('resolution_transparency', 'enum:transparent,partial,opaque', 'How clearly the outcome was explained',
      'transparent = clear what was done or why not; partial = vague on timing or reasons; opaque = templated.'),
    *EJ('resolution_transparency', 'the transparency rating'),
    F('customer_sentiment_end', 'enum:positive,neutral,negative,no_reply',
      "Customer's sentiment in their last turn",
      "Sentiment of the customer's last own turn after the first agent reply; no_reply if none."),
    *EJ('customer_sentiment_end', 'the end sentiment'),
    F('deadline_pressure', 'bool', 'Customer expressed a deadline or high stakes',
      'true if the customer cites a deadline, evaluation, promotion, grant, ranking or similar stakes.'),
    *EJ('deadline_pressure', 'the deadline-pressure value'),
]
print(f'{len(fields_rows)} fields')

# ---- taxonomy sheet ------------------------------------------------------
tax_rows = []
for _l1, _leaves in JOURNEY.items():
    tax_rows.append({'module_name': MOD, 'level': 'L1', 'label': _l1, 'description': None, 'notes': None})
    for _l2, _d in _leaves.items():
        tax_rows.append({'module_name': MOD, 'level': 'L2', 'label': _l2,
                         'description': f'Definition: {_d}', 'notes': None})

modules_rows = [{
    'module_name': MOD, 'taxonomy_prefix': PREFIX, 'taxonomy_file': f'{MOD}.json',
    'reorder_group': 'A', 'reorder_order': 1, 'enabled': 'TRUE', 'description': MOD_DESC,
    'notes': 'Written by notebooks/17_scopus_2ls_escalation_effort.ipynb. Do not edit by hand.'}]

with pd.ExcelWriter(DEFN_XLSX, engine='openpyxl') as _w:
    pd.DataFrame(modules_rows).to_excel(_w, sheet_name='modules', index=False)
    pd.DataFrame(fields_rows).to_excel(_w, sheet_name='fields', index=False)
    pd.DataFrame(tax_rows).to_excel(_w, sheet_name='taxonomy', index=False)
print(f'Wrote {DEFN_XLSX.relative_to(BASE)}: 1 module, {len(fields_rows)} fields, {len(tax_rows)} taxonomy rows')

In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--force', MOD], 'build_pipeline')

cfg = json.loads(CONFIG_JSON.read_text(encoding='utf-8'))
assert set(cfg['modules']) == {MOD}, f'config modules: {list(cfg["modules"])}'
assert (MODULES_DIR / f'{MOD}.py').exists(), f'missing generated/modules/{MOD}.py'
_tax = json.loads((TAXONOMY_DIR / f'{MOD}.json').read_text(encoding='utf-8'))
JOURNEY_LABELS = {code: leaf['label'] for node in _tax.values()
                  for code, leaf in node.get('children', {}).items()}
JOURNEY_GROUP = {code: node['label'] for node in _tax.values() for code in node.get('children', {})}
assert len(JOURNEY_LABELS) == sum(len(v) for v in JOURNEY.values()), 'leaf count mismatch'

from rca_pipeline.utils import _STRUCTURAL_PREFIXES
_own = [r['field_name'] for r in fields_rows] + [c for c in features.columns if c != ID_COL] \
    + list(META_RENAME.values())
for _c in _own:
    for _bad in _STRUCTURAL_PREFIXES:
        assert not _c.startswith(_bad), f'column {_c!r} collides with structural prefix {_bad!r}'
print(f'\nBuild verified. Journey codes:')
for _code, _label in JOURNEY_LABELS.items():
    print(f'  {_code:<8} {JOURNEY_GROUP[_code]:<36} {_label}')

In [ ]:
import importlib

from rca_pipeline.taxonomy import taxonomy_to_rich_prompt

importlib.invalidate_caches()
_name = f'generated.modules.{MOD}'
_mod = importlib.reload(sys.modules[_name]) if _name in sys.modules else importlib.import_module(_name)
PROMPT = _mod._build_system_prompt(taxonomy_to_rich_prompt(_tax, PREFIX))

for _text, _what in [(G_ESCALATION, 'escalation definition'), (G_NOTES, 'internal-note definition'),
                     (G_EVIDENCE, 'evidence rules'), (G_FORMAT, 'transcript format')]:
    assert _text in PROMPT, f'{_what} missing from the prompt'
for _k in FRICTION:
    assert f"'{_k}'" in PROMPT, f'friction category {_k} missing from the prompt'

# Prompt + output-affecting tuning + preprocessing: change any, and the checkpoint is discarded.
_sig = '|'.join(f'{k}={TUNING[k]}' for k in OUTPUT_AFFECTING) + \
       f'|{TURN_CAP}|{TURN_HEAD}|{TRUNC_CAP}|{TRUNC_HEAD}|{TRUNC_TAIL}'
PROMPT_FINGERPRINT = hashlib.sha256((PROMPT + _sig).encode('utf-8')).hexdigest()[:16]
print(f'System prompt: {len(PROMPT):,} chars (~{len(PROMPT) // 4:,} tokens), fingerprint {PROMPT_FINGERPRINT}\n')
print(PROMPT)

## 5. Profile, stage runner and the gated runs

`rca run` silently ignores a profile key it does not recognise, so every key is checked
against `--help` before writing.

The pipeline resumes from `runs/checkpoints/<input stem>_checkpoint.jsonl` and skips any ticket
already done. After a prompt edit that would silently return the old answers, so a fingerprint of
the prompt and the tuning is stored next to the checkpoint, and a mismatch discards it.

In [ ]:
_help = subprocess.run([sys.executable, '-m', 'rca_pipeline.cli', 'run', '--help'],
                       text=True, encoding='utf-8', errors='replace', capture_output=True,
                       env=SUBPROCESS_ENV)
assert _help.returncode == 0, _help.stderr[-2000:]
_dests = {x.replace('-', '_') for x in re.findall(r'--([a-z][a-z0-9-]+)', _help.stdout)}

spec = {'_comment': ('Scopus Content 2LS: escalation check, back-and-forth count, internal notes and '
                     'customer effort, with evidence. Input is the deduplicated transcript rebuilt from the '
                     'raw thread TSV by notebooks/17_scopus_2ls_escalation_effort.ipynb.'),
        'input': SAMPLE_INPUT.relative_to(BASE).as_posix(),
        'output': SAMPLE_OUT.relative_to(BASE).as_posix(),
        'taxonomy_dir': 'generated/taxonomies', 'text_col': TEXT_COL, 'id_col': ID_COL,
        'modules': [MOD], **TUNING}
_unknown = [k for k in spec if k != '_comment' and k not in _dests]
assert not _unknown, f'keys rca run would silently ignore: {_unknown}'
PROFILE.write_text(json.dumps(spec, indent=2) + '\n', encoding='utf-8')
print(PROFILE.read_text(encoding='utf-8'))


def checkpoint_files(input_path) -> list:
    stem = Path(input_path).stem
    return [CHECKPOINT_DIR / f'{stem}_checkpoint.jsonl', CHECKPOINT_DIR / f'{stem}_bad_uids.txt']


def _fingerprint_file(input_path) -> Path:
    return CHECKPOINT_DIR / f'{Path(input_path).stem}_checkpoint.fingerprint'


def _sync_checkpoint(input_path) -> None:
    fp = _fingerprint_file(input_path)
    stored = fp.read_text(encoding='utf-8').strip() if fp.exists() else None
    if stored == PROMPT_FINGERPRINT:
        return
    dropped = [p.name for p in checkpoint_files(input_path) if p.exists()]
    for p in checkpoint_files(input_path):
        p.unlink(missing_ok=True)
    if dropped:
        print(f'  prompt changed since that checkpoint ({stored or "no fingerprint"} -> '
              f'{PROMPT_FINGERPRINT}); discarded {", ".join(dropped)}')


def _stamp_checkpoint(input_path) -> None:
    _fingerprint_file(input_path).write_text(PROMPT_FINGERPRINT, encoding='utf-8')


def run_stage(input_path, output_path, label, fresh=False) -> Path:
    """Run the module as a subprocess (a dropped kernel does not kill a long run)."""
    if fresh:
        for p in [*checkpoint_files(input_path), _fingerprint_file(input_path)]:
            p.unlink(missing_ok=True)
    else:
        _sync_checkpoint(input_path)
    cmd = [sys.executable, '-m', 'rca_pipeline.cli', 'run', '--profile', str(PROFILE),
           '--input', str(input_path), '--output', str(output_path), '--modules', MOD]
    print(f'\n{"=" * 78}\n{label}\n{"=" * 78}\n$ {" ".join(cmd)}\n')
    _t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            encoding='utf-8', errors='replace', bufsize=1, env=SUBPROCESS_ENV)
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'{label} failed (exit {rc})')
    assert Path(output_path).exists(), f'{label} reported success but wrote no output'
    _stamp_checkpoint(input_path)
    print(f'\n{label}: {time.time() - _t0:.0f}s -> {Path(output_path).relative_to(BASE)}')
    return Path(output_path)


def latest_cost_report() -> dict:
    reps = sorted((BASE / TUNING['cost_report_dir']).glob('costs_*.json'))
    return json.loads(reps[-1].read_text(encoding='utf-8'))['summary'] if reps else {}


def write_input(df, path) -> None:
    df[[ID_COL, TEXT_COL]].to_csv(path, index=False, encoding='utf-8')
    back = pd.read_csv(path, dtype=str)
    assert list(back.columns) == [ID_COL, TEXT_COL] and len(back) == len(df), f'{path.name} did not round-trip'
    print(f'{path.relative_to(BASE)}: {len(df):,} tickets')


print('\nStage runner and checkpoint guard defined.')

In [ ]:
# ---- Parse a stage output into typed, analysable columns -----------------
INT_FIELDS  = ['escalation_count', 'return_count', 'customer_chase_count']
BOOL_FIELDS = ['customer_repeat_info', 'deadline_pressure']


def _slots(df, base, key):
    pat = re.compile(rf'^{re.escape(base)}_{key}_(\d+)$')
    return sorted((c for c in df.columns if pat.match(c)), key=lambda c: int(pat.match(c).group(1)))


def load_llm(path) -> pd.DataFrame:
    df = read_stage_output(path)
    s = lambda n: f'{STUB}_{n}'
    for n in INT_FIELDS:
        df[s(n)] = pd.to_numeric(df.get(s(n)), errors='coerce').astype('Int64')
    for n in BOOL_FIELDS:
        df[s(n)] = df.get(s(n)).astype(str).str.strip().str.lower().map(
            {'true': True, 'false': False, '1': True, '0': False, '1.0': True, '0.0': False}).astype('boolean')
    df['s2_journey'] = df[s('journey_code')].map(JOURNEY_LABELS)
    df['s2_journey_group'] = df[s('journey_code')].map(JOURNEY_GROUP)

    # Hand-off events, recounted from the list so the counts can be checked
    ev = s('handoff_events')
    dirs, evts = _slots(df, ev, 'direction'), _slots(df, ev, 'event')
    D = df[dirs].fillna('').astype(str).apply(lambda c: c.str.strip().str.lower()) if dirs else pd.DataFrame(index=df.index)
    E = df[evts].fillna('').astype(str).apply(lambda c: c.str.strip().str.lower()) if evts else pd.DataFrame(index=df.index)
    df['s2_events_n'] = (E != '').sum(axis=1)
    df['s2_events_to_2ls_handoffs'] = E.isin(['escalated', 're_escalated']).sum(axis=1)
    df['s2_events_from_2ls'] = (D == 'from_2ls').sum(axis=1)
    df['s2_events_chased_2ls'] = (E == 'chased_2ls').sum(axis=1)
    df['s2_events_unknown_type'] = ((E != '') & ~E.isin(EVENT_TYPES)).sum(axis=1)
    df['s2_events_sequence'] = E.apply(lambda r: ' > '.join(v for v in r if v), axis=1) if evts else ''
    df['s2_count_mismatch'] = (df[s('escalation_count')].fillna(-1) != df['s2_events_to_2ls_handoffs']) | \
                              (df[s('return_count')].fillna(-1) != df['s2_events_from_2ls'])
    # The counts used downstream are recounted from the event list, where every
    # event carries its own quote. A ticket judged escalated only because a 2LS
    # answer was relayed has no hand-off event, so it is floored at 1.
    df['n_handoffs_to_2ls'] = np.where(df[s('escalated')].eq('yes'),
                                       df['s2_events_to_2ls_handoffs'].clip(lower=1),
                                       df['s2_events_to_2ls_handoffs'])
    df['n_returns_from_2ls'] = df['s2_events_from_2ls']
    # Re-escalations: hand-offs after the first. This is the "bounce" count.
    df['n_reescalations'] = (df['n_handoffs_to_2ls'] - 1).clip(lower=0)

    # Friction points -> one boolean column per category
    fr = s('friction_points')
    cats = _slots(df, fr, 'category')
    C = df[cats].fillna('').astype(str).apply(lambda c: c.str.strip().str.lower()) if cats else pd.DataFrame(index=df.index)
    for k in FRICTION:
        df[f'fr_{k}'] = (C == k).any(axis=1)
    df['s2_friction_n'] = (C != '').sum(axis=1)
    df['s2_friction_unknown'] = C.apply(lambda r: ', '.join(v for v in r if v and v not in FRICTION), axis=1) if cats else ''
    df['s2_friction_list'] = C.apply(lambda r: ', '.join(v for v in r if v), axis=1) if cats else ''
    return df


def grounded_rate(df) -> pd.Series:
    """Share of rows per *_evidence column whose quote was found in the transcript.

    Empty evidence counts as grounded; a fabricated quote is blanked and flagged False."""
    g = [c for c in df.columns if c.endswith('_grounded')]
    if not g:
        return pd.Series(dtype=float)
    return df[g].apply(lambda c: c.astype(str).str.lower().isin(['true', '1', '1.0'])).mean().sort_values()


def qa_report(df, label) -> None:
    n = len(df)
    print(f'--- {label}: {n:,} tickets ---')
    print(f'blank s2_escalated     : {df["s2_escalated"].fillna("").eq("").sum()}')
    print(f'blank / unknown journey: {df["s2_journey"].isna().sum()}')
    print(f'count vs event mismatch: {df["s2_count_mismatch"].sum()} ({df["s2_count_mismatch"].mean():.1%})')
    print(f'unknown event types    : {int((df["s2_events_unknown_type"] > 0).sum())}')
    print(f'unknown friction labels: {int(df["s2_friction_unknown"].astype(str).ne("").sum())}')
    _inc = (df['s2_escalated'].eq('yes') & df['s2_journey_group'].eq('Not escalated')) | \
           (df['s2_escalated'].eq('no') & df['s2_journey_group'].fillna('').str.startswith('Escalated'))
    print(f'escalated vs journey contradictions: {int(_inc.sum())}')
    gr = grounded_rate(df)
    if len(gr):
        print(f'evidence grounded (worst 5 of {len(gr)} columns):\n{gr.head(5).round(3).to_string()}')
    print(f'\ns2_escalated:\n{df["s2_escalated"].value_counts(dropna=False).to_string()}')
    print(f'\njourney:\n{df["s2_journey"].value_counts(dropna=False).to_string()}')


print('Parsers defined.')

### The LLM pool is the sample

The smoke test draws from inside the sample, so its tickets are paid for once and reused by the sample
run. The smoke strata are the regex escalation flag × transcript length tercile. That way the smoke test
sees both short one-shot tickets and long multi-round ones, not just the typical web-form ticket.

In [ ]:
pool = (sample[[ID_COL, 'cell', 'year', 'cr1_rep', 'rc_rep', 'weight']]
        .merge(transcripts[[ID_COL, TEXT_COL, 'transcript_chars', 'was_truncated']], on=ID_COL)
        .merge(features[[ID_COL, 'rx_escalated', 'rx_escalation_episodes', 'rx_internal_note_turns']], on=ID_COL))
assert len(pool) == len(sample)
pool['len_band'] = pd.qcut(pool['transcript_chars'], 3, labels=['short', 'mid', 'long'])
pool['stratum'] = pool['rx_escalated'].map({True: 'rx_esc', False: 'rx_none'}) + '/' + pool['len_band'].astype(str)
print(f'LLM pool (= the sample): {len(pool):,} tickets')
print(pool['stratum'].value_counts().sort_index().to_string())

## Gate 1: smoke test *(~$1.30, always runs)*

40 tickets with real LLM calls (`fresh=True`: a smoke test that replays a checkpoint proves
nothing). The strata are topped up with multi-episode tickets and tickets with a note-like staff entry, so
the back-and-forth and internal-note paths both get exercised.

**Read the review workbook** (`runs/smoke_s2ls/s2ls_smoke_review.xlsx`): each row puts the transcript
next to every judgement, evidence and justification.

In [ ]:
_is_multi = pool['rx_escalation_episodes'] >= 2
_multi = pool[_is_multi].sample(n=min(6, int(_is_multi.sum())), random_state=SEED)
_is_note = (pool['rx_internal_note_turns'] > 0) & ~pool[ID_COL].isin(set(_multi[ID_COL]))
_notes = pool[_is_note].sample(n=min(4, int(_is_note.sum())), random_state=SEED)
_seeded = pd.concat([_multi, _notes])
smoke = pd.concat([_seeded, stratified_sample(pool, 'stratum', SMOKE_N - len(_seeded), exclude=_seeded[ID_COL])])
assert smoke[ID_COL].is_unique and len(smoke) == SMOKE_N
write_input(smoke, SMOKE_INPUT)
print(smoke['stratum'].value_counts().sort_index().to_string())

smoke_res = load_llm(run_stage(SMOKE_INPUT, SMOKE_OUT, f'SMOKE - {len(smoke)} tickets', fresh=True))
assert set(smoke_res[ID_COL]) == set(smoke[ID_COL]), 'smoke output id set differs from its input'
qa_report(smoke_res, 'smoke')
_c = latest_cost_report()
if _c:
    print(f'\nCost: ${_c.get("total_cost_usd", 0):.2f} for {_c.get("total_records")} records '
          f'(${_c.get("avg_cost_per_record", 0):.4f}/ticket, '
          f'{_c.get("total_completion_tokens", 0) / max(_c.get("total_records", 1), 1):,.0f} output tokens/ticket)')

In [ ]:
# ---- Side-by-side review workbook --------------------------------------
_front = [ID_COL, 's2_escalated', 's2_journey', 'n_handoffs_to_2ls', 'n_returns_from_2ls', 's2_events_sequence',
          's2_count_mismatch', 's2_internal_note_type', 's2_effort_score', 's2_friction_list',
          'rx_escalated', 'rx_escalation_episodes']
review = (smoke_res.merge(features[[ID_COL, 'rx_escalated', 'rx_escalation_episodes', 'obs_customer_turns',
                                    'obs_span_days']], on=ID_COL)
          .merge(smoke[[ID_COL, TEXT_COL]], on=ID_COL, suffixes=('_out', '')))
review = review[_front + [c for c in review.columns if c.startswith(('s2_', 'n_')) and c not in _front] + [TEXT_COL]]
review.to_excel(SMOKE_REVIEW, index=False)
print(f'Wrote {SMOKE_REVIEW.relative_to(BASE)}\n')

print('LLM vs regex (rows: s2_escalated, cols: rx_escalated):')
print(pd.crosstab(review['s2_escalated'], review['rx_escalated'], margins=True).to_string())

_note_ids = set(_notes[ID_COL])
print(f'\nInternal-note tickets seeded into the smoke: {len(_note_ids)}; model found a note in '
      f'{int(review.loc[review[ID_COL].isin(_note_ids), "s2_internal_note_type"].ne("none").sum())}')
for _, r in review[review[ID_COL].isin(_note_ids)].iterrows():
    print(f'  [{r[ID_COL]}] {r["s2_internal_note_type"]}: {str(r.get("s2_internal_note_summary", ""))[:200]}')
    print(f'      evidence: {str(r.get("s2_internal_note_evidence", ""))[:200]}')

for _, r in review.head(6).iterrows():
    print(f'\n{"-" * 78}\n[{r[ID_COL]}] escalated={r["s2_escalated"]}  journey={r["s2_journey"]}  '
          f'to2ls={r["n_handoffs_to_2ls"]} from2ls={r["n_returns_from_2ls"]}  effort={r["s2_effort_score"]}')
    print(f'  events   : {r["s2_events_sequence"]}')
    print(f'  evidence : {str(r.get("s2_escalated_evidence", ""))[:200]}')
    print(f'  why      : {r.get("s2_escalated_justification", "")}')
    print(f'  note     : {r["s2_internal_note_type"]} - {str(r.get("s2_internal_note_summary", ""))[:160]}')
    print(f'  friction : {r["s2_friction_list"]}')
    print(f'  effort   : {r.get("s2_effort_justification", "")}')

### Decision point

The sample run costs ~$180 and takes ~2.7 hours. Set `RUN_SAMPLE = True` in section 0 when:

- the `escalated` decisions agree with your own reading of the transcript;
- the event lists and counts match the transcript, and `count vs event mismatch` is near zero;
- the evidence is grounded, with `*_grounded` near 1.0;
- the justifications reason from named turns rather than restating the value.

If a definition needs changing, edit it in section 4 and re-run from there. The fingerprint
guard discards stale checkpoints.

In [ ]:
if not RUN_SAMPLE:
    raise SystemExit('RUN_SAMPLE is False - read the smoke review, then set RUN_SAMPLE = True in section 0.')
print('Continuing to the sample run.')

## 5b. The sample run

It runs as a subprocess, so a dropped kernel does not kill it. If it is interrupted, **re-run this
cell**: the checkpoint resumes and only the unfinished tickets are paid for. The 40 smoke tickets are
seeded into the checkpoint so they are not paid for twice. That is safe only while the prompt
fingerprint matches, and the code checks it.

Afterwards the cell reports:
- agreement between the LLM and the regex;
- whether effort tracks the observed measures (Spearman correlation);
- grounding;
- the actual cost.

In [ ]:
from scipy.stats import spearmanr

write_input(pool, SAMPLE_INPUT)
_sync_checkpoint(SAMPLE_INPUT)
_src, _dst = checkpoint_files(SMOKE_INPUT)[0], checkpoint_files(SAMPLE_INPUT)[0]
_smoke_fp = _fingerprint_file(SMOKE_INPUT)
if (not _dst.exists() and _src.exists() and _smoke_fp.exists()
        and _smoke_fp.read_text(encoding='utf-8').strip() == PROMPT_FINGERPRINT):
    shutil.copy2(_src, _dst)
    _stamp_checkpoint(SAMPLE_INPUT)
    print(f'Seeded {_dst.name} with the {SMOKE_N} smoke tickets')

llm_res = load_llm(run_stage(SAMPLE_INPUT, SAMPLE_OUT, f'SAMPLE - {len(pool):,} tickets'))
assert set(llm_res[ID_COL]) == set(pool[ID_COL]), 'sample output id set differs from its input'
qa_report(llm_res, 'sample')

_p = llm_res.merge(features, on=ID_COL)
print('\nLLM vs regex (unweighted):')
print(pd.crosstab(_p['s2_escalated'], _p['rx_escalated'], margins=True).to_string())
print(f'agreement on escalated yes/no: {((_p["s2_escalated"] == "yes") == _p["rx_escalated"]).mean():.1%}')
print('\nLLM hand-offs (from events) vs regex episodes:')
print(pd.crosstab(_p['n_handoffs_to_2ls'].clip(upper=3), _p['rx_escalation_episodes'].clip(upper=3)).to_string())

_EFF = {'very_low': 1, 'low': 2, 'medium': 3, 'high': 4, 'very_high': 5}
_e = _p['s2_effort_score'].map(_EFF)
print('\nSpearman: effort_score vs observed')
for c in ['obs_customer_turns', 'obs_customer_followups', 'obs_span_days', 'obs_max_wait_days',
          'obs_agent_changes', 'rx_customer_chasers']:
    _m = _e.notna() & _p[c].notna()
    rho, _ = spearmanr(_e[_m], _p.loc[_m, c])
    print(f'  {c:<24} rho={rho:+.2f}  (n={_m.sum()})')

_c = latest_cost_report()
if _c:
    print(f'\nCost of this run: ${_c.get("total_cost_usd", 0):,.2f} for {_c.get("total_records")} tickets '
          f'(${_c.get("avg_cost_per_record", 0):.4f}/ticket)')

print('\n--- to read: LLM "no" where the regex saw hand-off wording ---')
for _, r in _p[(_p['s2_escalated'] == 'no') & _p['rx_escalated']].head(5).iterrows():
    print(f'  [{r[ID_COL]}] {r["s2_journey"]} - {r.get("s2_escalated_justification", "")}')
print('\n--- LLM "yes" where the regex saw nothing ---')
for _, r in _p[(_p['s2_escalated'] == 'yes') & ~_p['rx_escalated']].head(5).iterrows():
    print(f'  [{r[ID_COL]}] {str(r.get("s2_escalated_evidence", ""))[:160]}')

## 6. Merge, weight, summarise, write the deliverable

One row per **sampled** ticket:
- metadata + `obs_*` + `rx_*`;
- every LLM judgement with its evidence, justification and `*_grounded` flag;
- its `cell`, `weight` and `reportable` flag.

The free `obs_*` / `rx_*` features for all 63,824 tickets stay in `runs/intermediate/s2ls_rule_features.csv`.

**Every population figure is weighted.** `est_tickets` is the estimated number of tickets in the full
population; `%` columns are weighted shares; `sampled` is how many tickets were actually read.

**The back-and-forth counts:**
- `n_handoffs_to_2ls`: 1LS → 2LS, the first hand-off plus every re-escalation.
- `n_returns_from_2ls`: 2LS → 1LS, i.e. answers, rejections, information requests, completions.
- `n_reescalations`: hand-offs after the first.

They are recounted from the event list, where every event has its own quote. The model's own
`s2_escalation_count` / `s2_return_count` are kept next to them, and `s2_count_mismatch` flags where the
two disagree.

**`by_year_contact_resolution`** has one row per cell, with a 95% Wilson interval on % escalated.
Within a cell every ticket has the same weight, so the cell figures are plain sample proportions.

In [ ]:
_llm_cols = [c for c in llm_res.columns if c.startswith(('s2_', 'n_', 'fr_')) and c != TEXT_COL]
final = (sample[[ID_COL, 'cell', 'year', 'cr1_rep', 'rc_rep', 'weight']]
         .merge(alloc[['cell', 'reportable']], on='cell', validate='m:1')
         .merge(meta, on=ID_COL, validate='1:1')
         .merge(transcripts[[ID_COL, 'transcript_chars', 'was_truncated']], on=ID_COL, validate='1:1')
         .merge(features, on=ID_COL, validate='1:1')
         .merge(llm_res[[ID_COL] + _llm_cols], on=ID_COL, how='left', validate='1:1'))
final['s2_escalated'] = final['s2_escalated'].fillna('').replace('', 'unclear')
assert final[ID_COL].is_unique
print(f'Deliverable: {len(final):,} sampled tickets x {final.shape[1]} columns, '
      f'weights sum to {final["weight"].sum():,.0f}')

_EFF_ORDER = ['very_low', 'low', 'medium', 'high', 'very_high']


def _bucket(s):
    return s.astype('Float64').clip(upper=3).map(lambda v: '' if pd.isna(v) else ('3+' if v >= 3 else str(int(v))))


def wdist(df, by, col):
    """Weighted % of each `col` value within each `by` group, with est. population and sample size."""
    d = df.assign(_v=df[col].fillna('(blank)').astype(str))
    t = d.pivot_table(index=by, columns='_v', values='weight', aggfunc='sum', fill_value=0, observed=True)
    out = t.div(t.sum(axis=1), axis=0).mul(100).round(1).add_suffix(' %')
    out.insert(0, 'sampled', d.groupby(by).size())
    out.insert(0, 'est_tickets', t.sum(axis=1).round().astype(int))
    return out.sort_values('est_tickets', ascending=False)


def wtotal(df, col):
    """Weighted distribution of `col` over the whole population."""
    d = df.assign(_v=df[col].fillna('(blank)').astype(str))
    t = d.groupby('_v').agg(est_tickets=('weight', 'sum'), sampled=('weight', 'size'))
    t['share_%'] = (t['est_tickets'] / t['est_tickets'].sum() * 100).round(1)
    t['est_tickets'] = t['est_tickets'].round().astype(int)
    return t.sort_values('est_tickets', ascending=False).rename_axis(col)


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(100 * (c - h), 1), round(100 * (c + h), 1))


# ---- Weighted population estimates ----------------------------------------
_fr_cols = [f'fr_{k}' for k in FRICTION]
_fr = pd.DataFrame({'share_%': (final[_fr_cols].mul(final['weight'], axis=0).sum() / final['weight'].sum() * 100)
                    .round(1), 'sampled_with': final[_fr_cols].sum()}).rename(index=lambda c: c[3:])
_fr = _fr.sort_values('share_%', ascending=False)
_fr['definition'] = _fr.index.map(FRICTION)

_trips = final.assign(**{'hand-offs 1LS->2LS': _bucket(final['n_handoffs_to_2ls']),
                         'returns 2LS->1LS': _bucket(final['n_returns_from_2ls'])})
_back = (_trips.pivot_table(index='hand-offs 1LS->2LS', columns='returns 2LS->1LS', values='weight',
                            aggfunc='sum', fill_value=0) / final['weight'].sum() * 100).round(1)

_eff_obs = final.groupby('s2_effort_score')[['obs_customer_turns', 'obs_customer_followups', 'obs_span_days',
                                              'obs_max_wait_days', 's2_customer_chase_count']] \
    .median().reindex(_EFF_ORDER).round(1)

# ---- One row per year x contact reason x resolution cell -------------------
_rows = []
for (yr, cr, rc), g in final.groupby(['year', 'cr1_rep', 'rc_rep'], sort=True):
    n, k = len(g), int(g['s2_escalated'].eq('yes').sum())
    lo, hi = wilson(k, n)
    _top = g[_fr_cols].mean()
    _rows.append({
        'year': yr, 'contact_reason': cr, 'resolution_code': rc,
        'population': int(round(g['weight'].sum())), 'sampled': n,
        'reportable': bool(g['reportable'].iloc[0]),
        'escalated_%': round(100 * k / n, 1), 'escalated_ci_low': lo, 'escalated_ci_high': hi,
        'mean_handoffs_to_2ls': round(g['n_handoffs_to_2ls'].mean(), 2),
        'mean_returns_from_2ls': round(g['n_returns_from_2ls'].mean(), 2),
        're_escalated_%': round(100 * g['n_reescalations'].gt(0).mean(), 1),
        'internal_note_%': round(100 * g['s2_internal_note_type'].fillna('none').ne('none').mean(), 1),
        'effort_high_or_very_high_%': round(100 * g['s2_effort_score'].isin(['high', 'very_high']).mean(), 1),
        'top_friction': _top.idxmax()[3:] if _top.max() > 0 else '',
        'top_friction_%': round(100 * _top.max(), 1),
    })
by_cell = pd.DataFrame(_rows).sort_values(['year', 'population'], ascending=[True, False]).reset_index(drop=True)

SUMMARY = {
    'escalated': wtotal(final, 's2_escalated'),
    'by_year': wdist(final, 'year', 's2_escalated'),
    'by_contact_reason': wdist(final, 'cr1_rep', 's2_escalated'),
    'by_resolution_code': wdist(final, 'rc_rep', 's2_escalated'),
    'by_year_contact_resolution': by_cell,
    'back_and_forth_%': _back,
    'journey': wtotal(final, 's2_journey'),
    'escalation_target': wtotal(final, 's2_escalation_target'),
    'internal_notes': wtotal(final, 's2_internal_note_type'),
    'friction': _fr,
    'effort': wtotal(final, 's2_effort_score'),
    'effort_by_handoffs': wdist(_trips, 'hand-offs 1LS->2LS', 's2_effort_score'),
    'effort_by_year': wdist(final, 'year', 's2_effort_score'),
    'outcome': wtotal(final, 's2_resolution_outcome'),
    'effort_vs_observed': _eff_obs,
    'sampling_allocation': alloc.drop(columns='cell'),
}
for k, v in SUMMARY.items():
    print(f'\n=== {k} ===\n{v.head(15).to_string()}')
print(f'\nReportable cells: {int(by_cell["reportable"].sum())} of {len(by_cell)}')

In [ ]:
# ---- Column order: identity, sampling, verdicts, counts, notes, effort, then the rest ----
_lead = [ID_COL, 'year', 'cr1_rep', 'rc_rep', 'cell', 'weight', 'reportable',
         'Channel', 'Chat_Start_Date', 'rc_lvl1', 'cr1_desc', 'cr2_desc', 'customer_classification_type',
         's2_escalated', 's2_escalated_evidence', 's2_escalated_justification',
         's2_escalation_target', 's2_journey_group', 's2_journey', 's2_journey_code', 's2_journey_justification',
         'n_handoffs_to_2ls', 'n_returns_from_2ls', 'n_reescalations', 's2_events_sequence',
         's2_escalation_count', 's2_return_count',
         's2_handoff_justification', 's2_count_mismatch', 's2_not_escalated_reason',
         's2_internal_note_type', 's2_internal_note_evidence', 's2_internal_note_justification',
         's2_internal_note_summary', 's2_effort_score', 's2_effort_evidence', 's2_effort_justification',
         's2_friction_list', 's2_friction_n']
_lead = [c for c in _lead if c in final.columns]
_rest = [c for c in final.columns if c not in _lead]
_order = _lead + [c for c in _rest if c.startswith(('s2_', 'n_'))] + [c for c in _rest if c.startswith('fr_')] \
    + [c for c in _rest if c.startswith('obs_')] + [c for c in _rest if c.startswith('rx_')] \
    + [c for c in _rest if not c.startswith(('s2_', 'n_', 'fr_', 'obs_', 'rx_'))]
final = final[_order]

_no = final[final['s2_escalated'].eq('no')]
_qa = (_no.sample(n=min(100, len(_no)), random_state=SEED)
       .merge(transcripts[[ID_COL, 'transcript_full']], on=ID_COL))
_qa['transcript_full'] = _qa['transcript_full'].str[:32_000]     # Excel cell limit

with pd.ExcelWriter(FINAL_XLSX, engine='openpyxl') as _w:
    final.to_excel(_w, sheet_name='tickets', index=False)
    for k, v in SUMMARY.items():
        v.to_excel(_w, sheet_name=k[:31], index=not isinstance(v.index, pd.RangeIndex))
    _qa.to_excel(_w, sheet_name='not_escalated_sample', index=False)
print(f'Wrote {FINAL_XLSX.relative_to(BASE)}  ({len(final):,} tickets, {len(SUMMARY) + 2} sheets)')

final.merge(transcripts[[ID_COL, 'transcript_full']], on=ID_COL).to_csv(FINAL_CSV, index=False, encoding='utf-8')
print(f'Wrote {FINAL_CSV.relative_to(BASE)}')

## 7. Put the previous product's config back *(optional)*

The build replaced `generated/pipeline_config.json` with this notebook's module. To go back to what
was live before, run the command recorded in `generated/_backup_pre_scopus_2ls/RESTORE.txt`.

In [ ]:
print((CONFIG_BACKUP / 'RESTORE.txt').read_text(encoding='utf-8'))